# Build Object A (human-only)

* **Developed by:** Anna Maguza
* **Affilation:** CellZome, a GSK company
* **Created date:** 2026-05-08
* **Last modified date:** 2026-05-13

Concat atlas + Ishikawa 2022 → object_a_human.h5ad for 3a notebooks.


## Goal

Build **Object A** (human-only): concat the atlas (epithelial subset, 0a) with the standardised Ishikawa 2022 object (0b). Common gene space = inner intersection of `var_names`. Save as `object_a_human.h5ad` for use by 3a integration notebooks.

In [1]:
import os, sys, gc, json, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.verbosity = 2
sc.settings.dpi = 120
sc.settings.dpi_save = 300
plt.rcParams.update({
    'savefig.bbox': 'tight', 'savefig.dpi': 300, 'figure.dpi': 120,
    'font.family': ['Arial', 'Helvetica', 'DejaVu Sans'], 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

REPO         = Path('/Users/am336941/Library/CloudStorage/OneDrive-GSK/Desktop/Fetal_stem_cells_analysis')
DATA_OUT     = Path('/Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced')
ATLAS_PATH   = Path('/Users/am336941/PhD/data/gut_data/gut_hs_all_datasets_full_annotated_AM_30102025_181544_raw.h5ad')
LGR5_DIR     = REPO / 'data' / 'LGR5_analysis'
ORTH_PATH    = Path('/Users/am336941/PhD/data/LGR5_analysis_data/human_mouse_orthologues_ensembl.txt')
DATA_OUT.mkdir(parents=True, exist_ok=True)

def stamp() -> str:
    return dt.datetime.now().strftime('%d%m%Y_%H%M%S')


/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
atlas_path = sorted(DATA_OUT.glob('gut_hs_atlas_epithelial_AM_*_raw.h5ad'))[-1]
ishi_path  = sorted(DATA_OUT.glob('gut_hs_Ishikawa2022_std_*_raw.h5ad'))[-1]
print('atlas:', atlas_path); print('ishikawa:', ishi_path)

atlas = sc.read_h5ad(atlas_path)
ishi  = sc.read_h5ad(ishi_path)
print('atlas:', atlas.shape, '/ ishi:', ishi.shape)
print('atlas obs cols:', list(atlas.obs.columns))
print('ishi  obs cols:', list(ishi.obs.columns))

# ---- Drop stale scVI state from the pre-built atlas ---------------------
# The all_epithelial atlas carries leftover state from a previous scVI/scANVI
# integration (Elementaite + Holloway). We must remove it before reusing the
# AnnData with scvi-tools or downstream integration tools.
for k in ['_scvi_batch', '_scvi_labels']:
    if k in atlas.obs.columns:
        del atlas.obs[k]
for k in ['X_pca', 'X_scVI', 'X_scANVI', 'X_umap', 'umap_uncorrected',
          '_scvi_extra_categorical_covs']:
    if k in atlas.obsm:
        del atlas.obsm[k]
for k in ['_scvi_manager_uuid', '_scvi_uuid', 'hvg', 'neighbors', 'umap',
          'pca']:
    if k in atlas.uns:
        del atlas.uns[k]
# Ensure raw counts are in a 'counts' layer (the all_epithelial atlas stores
# counts in .X as int64, and the existing 'counts' layer is identical).
if 'counts' not in atlas.layers:
    atlas.layers['counts'] = atlas.X.copy()


atlas: /Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced/gut_hs_atlas_epithelial_AM_13052026_144012_raw.h5ad
ishikawa: /Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced/gut_hs_Ishikawa2022_std_13052026_144051_raw.h5ad
atlas: (95357, 43704) / ishi: (7103, 33538)
atlas obs cols: ['cell_index', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_counts_mito', 'percent_ribo', 'n_counts_ribo', 'percent_hb', 'n_counts_hb', 'percent_top50', 'cell_passed_qc', 'qc_cluster', 'cluster_passed_qc', 'c

## Make var_names compatible (use gene symbols, no duplicates)

In [3]:
def normalise_var_names(a):
    # if var_names look like row indices, switch to a meaningful symbol column
    if a.var_names.astype(str).str.fullmatch(r'\d+').all():
        for col in ['gene_name', 'gene_symbols', 'feature_name', 'symbol', 'Gene']:
            if col in a.var.columns:
                a.var_names = a.var[col].astype(str).values
                break
    a.var_names_make_unique()
    return a

atlas = normalise_var_names(atlas)
ishi  = normalise_var_names(ishi)
print('atlas first 3 var:', list(atlas.var_names[:3]))
print('ishi  first 3 var:', list(ishi.var_names[:3]))

shared = sorted(set(atlas.var_names) & set(ishi.var_names))
print(f'shared genes: {len(shared):,}')


atlas first 3 var: ['WASH7P', 'ENSG00000238009', 'CICP27']
ishi  first 3 var: ['MIR1302-2HG', 'FAM138A', 'OR4F5']
shared genes: 20,112


## Concatenate (preserve atlas obs schema)

In [4]:
atlas_sub = atlas[:, shared].copy()
ishi_sub  = ishi[:,  shared].copy()

# minimal common obs schema
for a, sname in [(atlas_sub, 'human_atlas'), (ishi_sub, 'Ishikawa2022')]:
    if 'Study_name' not in a.obs.columns:
        a.obs['Study_name'] = sname
    if 'lgr5_status' not in a.obs.columns:
        a.obs['lgr5_status'] = 'unknown'
    if 'sample_id' not in a.obs.columns:
        a.obs['sample_id'] = a.obs.get('sample', a.obs.get('Donor_ID', 'unknown')).astype(str)
    a.obs['species'] = 'human'

In [5]:
mapping={
            'Colono_1': 'Colonocyte', 'Colono_2': 'Colonocyte',
            'BEST4': 'BEST4+ epithelial',
            'Goblet_1': 'Goblet', 'Goblet_2': 'Goblet',
            'TA_1': 'TA', 'TA_2': 'TA', 'TA_3': 'TA',
            'Sec.Prog': 'Secretory progenitor',
            'Stem': 'Stem cells',
        }

ishi_sub.obs['cell_states'] = ishi_sub.obs['celltype'].map(mapping)

In [6]:
ishi_sub.obs

,cell_id,celltype,lgr5_status,lgr5_label_raw,sample,condition,cell_type,tissue,GSE,organism,technology,assay_modality,study_id,Study_name,species,library_preparation_protocol,cell_states,sample_id
0,AAACCTGAGTACGCGA-1,TA_3,unknown,TA_3,Ishikawa2022,healthy ascending colon,TA_3,ascending colon,Ishikawa2022 (data shared by authors; not via ...,homo sapiens,10x Chromium,single-cell,Ishikawa2022,Ishikawa 2022 (10x),human,10x Chromium,TA,Ishikawa2022
1,AAACCTGCACCGGAAA-1,Goblet_1,unknown,Goblet_1,Ishikawa2022,healthy ascending colon,Goblet_1,ascending colon,Ishikawa2022 (data shared by authors; not via ...,homo sapiens,10x Chromium,single-cell,Ishikawa2022,Ishikawa 2022 (10x),human,10x Chromium,Goblet,Ishikawa2022
2,AAACCTGCAGACGCTC-1,Colono_2,unknown,Colono_2,Ishikawa2022,healthy ascending colon,Colono_2,ascending colon,Ishikawa2022 (data shared by authors; not via ...,homo sapiens,10x Chromium,single-cell,Ishikawa2022,Ishikawa 2022 (10x),human,10x Chromium,Colonocyte,Ishikawa2022
3,AAACCTGCATCATCCC-1,TA_3,unknown,TA_3,Ishikawa2022,healthy ascending colon,TA_3,ascending colon,Ishikawa2022 (data shared by authors; not via ...,homo sapiens,10x Chromium,single-cell,Ishikawa2022,Ishikawa 2022 (10x),human,10x Chromium,TA,Ishikawa2022
4,AAACCTGCATCTATGG-1,Goblet_1,unknown,Goblet_1,Ishikawa2022,healthy ascending colon,Goblet_1,ascending colon,Ishikawa2022 (data shared by authors; not via ...,homo sapiens,10x Chromium,single-cell,Ishikawa2022,Ishikawa 2022 (10x),human,10x Chromium,Goblet,Ishikawa2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7098,TTTGTCAGTGCAGTAG-1,Sec.Prog,unknown,Sec.Prog,Ishikawa2022,healthy ascending colon,Sec.Prog,ascending colon,Ishikawa2022 (data shared by authors; not via ...,homo sapiens,10x Chromium,single-cell,Ishikawa2022,Ishikawa 2022 (10x),human,10x Chromium,Secretory progenitor,Ishikawa2022
7099,TTTGTCAGTTGCTCCT-1,TA_2,unknown,TA_2,Ishikawa2022,healthy ascending colon,TA_2,ascending colon,Ishikawa2022 (data shared by authors; not via ...,homo sapiens,10x Chromium,single-cell,Ishikawa2022,Ishikawa 2022 (10x),human,10x Chromium,TA,Ishikawa2022
7100,TTTGTCAGTTTGTGTG-1,BEST4,unknown,BEST4,Ishikawa2022,healthy ascending colon,BEST4,ascending colon,Ishikawa2022 (data shared by authors; not via ...,homo sapiens,10x Chromium,single-cell,Ishikawa2022,Ishikawa 2022 (10x),human,10x Chromium,BEST4+ epithelial,Ishikawa2022
7101,TTTGTCATCAAACCGT-1,Stem,LGR5+,Stem,Ishikawa2022,healthy ascending colon,Stem,ascending colon,Ishikawa2022 (data shared by authors; not via ...,homo sapiens,10x Chromium,single-cell,Ishikawa2022,Ishikawa 2022 (10x),human,10x Chromium,Stem cells,Ishikawa2022


In [7]:
# Pad Ishikawa with atlas's extra obs columns so concat keeps them.
ATLAS_KEEP = ['age_group', 'gut_region', 'organism_part', 'celltype']
for col in ATLAS_KEEP:
    if col in atlas_sub.obs.columns and col not in ishi_sub.obs.columns:
        # Sensible defaults for the LGR5+ adult colonic dataset
        default = {'age_group': 'adult', 'gut_region': 'large intestine',
                   'organism_part': 'colon', 'celltype': 'Epithelial'}.get(col, 'unknown')
        ishi_sub.obs[col] = default

obj_a = ad.concat({'atlas': atlas_sub, 'ishikawa': ishi_sub},
                  axis=0, join='inner', label='source', merge='unique')
obj_a.obs_names_make_unique()
print('Object A:', obj_a.shape)
print('obs columns:', list(obj_a.obs.columns))
print(obj_a.obs.groupby(['source','Study_name'], observed=True).size().head(20))


Object A: (102460, 20112)
obs columns: ['organism', 'organism_part', 'cell_type', 'sample_id', 'Study_name', 'library_preparation_protocol', 'age_group', 'celltype', 'cell_states', 'gut_region', 'lgr5_status', 'species', 'source']
source    Study_name         
atlas     Elementaite_2021       54600
          Holloway_2021          40757
ishikawa  Ishikawa 2022 (10x)     7103
dtype: int64


In [8]:
# Ensure raw counts live in layers['counts'] for scVI/scANVI/scPoli.
# atlas.X is the integer count matrix, ishi.X likewise — copy after concat.
import numpy as np
sample = obj_a.X[:200].toarray() if hasattr(obj_a.X, 'toarray') else obj_a.X[:200]
assert np.allclose(sample, np.round(sample)), 'X is not integer counts — investigate before training scVI'
obj_a.layers['counts'] = obj_a.X.copy()

In [9]:
# Append processing history (per CLAUDE.md uns convention)
proc = obj_a.uns.get('processing_history', {})
if not isinstance(proc, dict):
    proc = {'previous': proc}
proc[f'object_a_build_{stamp()}'] = {
    'script': '0d_build_object_A.ipynb',
    'atlas_in': str(atlas_path.name),
    'ishi_in': str(ishi_path.name),
    'n_cells': int(obj_a.n_obs),
    'n_genes': int(obj_a.n_vars),
    'shared_gene_intersection': int(obj_a.n_vars),
}
obj_a.uns['processing_history'] = proc

In [10]:
out = DATA_OUT / 'object_a_human.h5ad'
obj_a.write_h5ad(out, compression='gzip')
print('saved', out, f'({out.stat().st_size/1e9:.2f} GB)')

saved /Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced/object_a_human.h5ad (0.89 GB)
